# 167. RLVR：可验证奖励、Group Advantage 与防 Reward Hacking 怎样实现？

> **面试问题：数学/代码任务为什么适合 RLVR？verifier、奖励、策略 loss、数据切分和发布门禁怎样设计？**

## 先给结论

RLVR 的优势是正确性信号可由确定程序产生，但 verifier 本身就是安全关键规格。必须先定义规范化答案和拒绝语义，再把 correctness 与 format/安全奖励分开，使用隐藏测试和对抗样本阻断 reward hacking；优化算法只是消费这份奖励，不能修复错误 verifier。

## 推荐的回答主线

1. 把题目、候选答案、解析器、verifier 版本、超时和失败原因写成明确合同。
2. 数学题用规范化/精确比较，代码题在隔离环境跑隐藏性质测试，解析失败给显式结果。
3. 对同 prompt 的样本计算 group-relative advantage，再对 response token 应用 clipped policy objective。
4. 按 verifier 难度、奖励稀疏、hack slice、污染和独立评测器做训练与发布门禁。

## 本 Notebook 的实现边界

Notebook 不执行任意字符串代码；代码 verifier 只接收受控 Python callable，以展示隐藏性质测试语义。真实实现必须使用进程/容器隔离、资源配额、无网络、只读文件系统和审计。

## 一手资料

- [Tülu 3](https://arxiv.org/abs/2411.15124)
- [DeepSeek-R1](https://arxiv.org/abs/2501.12948)
- [DeepSeekMath / GRPO](https://arxiv.org/abs/2402.03300)


In [ ]:
import hashlib
import math
import re
from dataclasses import dataclass
from fractions import Fraction

import numpy as np
import torch

# 受控样本覆盖正确、格式错误和数值错误三类结果。
PROMPT = "计算 3/4 + 5/6，并把最终答案放在 <answer> 中。"
candidates = ["推导略。<answer>19/12</answer>", "19/12", "<answer>1.58</answer>"]

assert "<answer>" in candidates[0]
assert len(candidates) == 3
assert Fraction(3, 4) + Fraction(5, 6) == Fraction(19, 12)


## 1. 先写 verifier 合同：解析失败不等于数值错误

训练日志必须区分 parse error、wrong answer、timeout 和 verifier error，否则无法判断模型能力还是基础设施问题。解析器应只抽取约定区域，避免候选在解释里塞多个答案来碰撞规则。


In [ ]:
@dataclass(frozen=True)
class VerifyResult:
    correct: bool
    parsed: bool
    reason: str
    normalized: str | None

ANSWER_RE = re.compile(r"<answer>\s*([^<>]+?)\s*</answer>")

def extract_answer(text):
    matches = ANSWER_RE.findall(text)
    if len(matches) != 1:
        return None
    return matches[0].strip()

# 只接受唯一、闭合的 answer 区域，多答案与无标签都明确拒绝。
assert extract_answer(candidates[0]) == "19/12"
assert extract_answer(candidates[1]) is None
assert extract_answer("<answer>1</answer><answer>2</answer>") is None


## 2. 数学 verifier：优先精确规范化，再谈容差

分数和整数可以用有理数精确比较；浮点容差要绑定题型，避免宽容差把错误答案判对。生产解析还应限制长度、分母、指数和解析时间，防止拒绝服务。


In [ ]:
def verify_fraction(text, expected):
    raw = extract_answer(text)
    if raw is None:
        return VerifyResult(False, False, "parse_error", None)
    try:
        value = Fraction(raw)
    except (ValueError, ZeroDivisionError):
        return VerifyResult(False, False, "invalid_number", None)
    normalized = f"{value.numerator}/{value.denominator}"
    return VerifyResult(value == expected, True, "ok" if value == expected else "wrong", normalized)

# 等价分数应通过；无格式和近似小数分别是解析失败与数值错误。
expected = Fraction(19, 12)
results = [verify_fraction(text, expected) for text in candidates]
assert results[0].correct and results[0].normalized == "19/12"
assert results[1].reason == "parse_error"
assert results[2].parsed and not results[2].correct


## 3. 代码 verifier：隐藏性质测试比公开样例更难投机

公开样例只能说明几个点。下面验证候选函数的排序性、排列保持和幂等性；真实系统还需要 subprocess/container、CPU/内存/时间限制、禁网、临时文件和系统调用策略，绝不能在训练进程直接 `exec` 模型文本。


In [ ]:
def verify_sort_callable(candidate_fn):
    hidden = [[], [2, 1], [3, -1, 3, 2], list(range(20, -1, -1))]
    try:
        for case in hidden:
            original = list(case)
            output = candidate_fn(list(case))
            if output != sorted(original) or sorted(output) != sorted(original):
                return VerifyResult(False, True, "hidden_test_failed", None)
            if candidate_fn(list(output)) != output:
                return VerifyResult(False, True, "not_idempotent", None)
    except Exception as exc:
        return VerifyResult(False, True, f"runtime:{type(exc).__name__}", None)
    return VerifyResult(True, True, "ok", "all_hidden_properties")

# 正确实现通过；只反转和只处理长度 2 的投机实现失败。
assert verify_sort_callable(lambda values: sorted(values)).correct
assert not verify_sort_callable(lambda values: values[::-1]).correct
assert not verify_sort_callable(lambda values: sorted(values) if len(values) <= 2 else values).correct


## 4. 奖励分解：correctness、format 与安全违规分别记录

把所有信号揉成一个数会掩盖 hack。可用 correctness 为主，format 给小权重，并对越权/超资源直接 hard fail；监控仍保留每个分量。格式奖励绝不能大到让“格式漂亮但答案错”胜过正确答案。


In [ ]:
def reward_components(result, safe=True, format_weight=0.05):
    correctness = float(result.correct)
    format_score = float(result.parsed)
    safety = 0.0 if safe else -1.0
    total = correctness + format_weight * format_score + safety
    return {"correct": correctness, "format": format_score, "safety": safety, "total": total}

# 正确答案得分最高；越权即使答案正确也被惩罚；格式分不能盖过正确性。
good = reward_components(results[0], safe=True)
wrong = reward_components(results[2], safe=True)
unsafe = reward_components(results[0], safe=False)
assert good["total"] > wrong["total"]
assert unsafe["total"] < good["total"]
assert wrong["format"] == 1.0 and wrong["correct"] == 0.0


## 5. Group-relative advantage：同题内比较，并处理零方差组

对同 prompt 采样 G 个响应后，可在组内中心化/标准化奖励，减少对 value model 的依赖。若全对或全错，标准差为零，直接除法会爆炸；常见策略是加 epsilon、跳过或改进采样难度。


In [ ]:
def group_advantages(rewards, eps=1e-6):
    rewards = np.asarray(rewards, dtype=float)
    centered = rewards - rewards.mean(axis=1, keepdims=True)
    scale = rewards.std(axis=1, keepdims=True)
    advantages = np.where(scale > eps, centered / np.maximum(scale, eps), 0.0)
    return advantages, scale[:, 0]

# 非退化组优势均值为零；全同奖励组返回零而不是 NaN。
reward_groups = np.array([[1.05, 0.05, 0.0, 1.05], [0.0, 0.0, 0.0, 0.0]])
advantages, scales = group_advantages(reward_groups)
assert np.allclose(advantages.mean(axis=1), 0.0)
assert np.allclose(advantages[1], 0.0)
assert scales[0] > 0 and scales[1] == 0


## 6. Token-level clipped policy loss：只监督 response 有效位置

importance ratio 用新旧策略对同一采样 token 的 log-prob 差计算，clip 限制单次更新；prompt、padding 和失败后的无效 token 必须 mask。下面展示最小 surrogate，不代替完整 KL、熵和分布式 rollout 管理。


In [ ]:
def clipped_policy_loss(new_logp, old_logp, advantage, response_mask, clip=0.2):
    # 先把 mask 外差值归零，避免 exp 溢出后再乘零产生 0*inf=NaN。
    safe_delta = torch.where(response_mask.bool(), new_logp - old_logp, torch.zeros_like(new_logp))
    ratio = torch.exp(safe_delta)
    adv = advantage[:, None].expand_as(ratio)
    unclipped = ratio * adv
    clipped = ratio.clamp(1 - clip, 1 + clip) * adv
    objective = torch.minimum(unclipped, clipped)
    return -(objective * response_mask).sum() / response_mask.sum(), ratio

# mask 外的极端 log-prob 不应影响 loss，且 ratio 恒为正。
old_logp = torch.tensor([[-1.0, -0.7, 0.0], [-0.5, 0.0, 0.0]])
new_logp = torch.tensor([[-0.9, -1.2, 99.0], [-0.4, -99.0, 99.0]], requires_grad=True)
response_mask = torch.tensor([[1, 1, 0], [1, 0, 0]], dtype=torch.float32)
adv = torch.tensor([1.0, -0.5])
policy_loss, ratio = clipped_policy_loss(new_logp, old_logp, adv, response_mask)
policy_loss.backward()
assert torch.isfinite(policy_loss)
assert (ratio > 0).all()
assert new_logp.grad[response_mask == 0].abs().sum().item() == 0.0


## 7. 奖励稀疏与课程：看有效学习组，而不只看平均 reward

若一组全错或全对，组内 advantage 没有排序信号。训练面板应报告 parse rate、pass rate、非零方差组占比、各难度 slice、verifier error 和 hack 命中；课程目标是把样本留在“可探索但未饱和”区间。


In [ ]:
def learning_signal_report(binary_reward_matrix):
    matrix = np.asarray(binary_reward_matrix, dtype=float)
    pass_rate = matrix.mean()
    informative = (matrix.max(1) > matrix.min(1)).mean()
    all_wrong = (matrix.sum(1) == 0).mean()
    all_right = (matrix.sum(1) == matrix.shape[1]).mean()
    return {"pass_rate": pass_rate, "informative_groups": informative, "all_wrong": all_wrong, "all_right": all_right}

# 混合组才提供组内排序信号，四类组比例应具有一致含义。
matrix = [[1, 0, 0, 1], [0, 0, 0, 0], [1, 1, 1, 1], [0, 1, 0, 0]]
report = learning_signal_report(matrix)
assert math.isclose(report["informative_groups"], 0.5)
assert math.isclose(report["all_wrong"], 0.25)
assert math.isclose(report["all_right"], 0.25)


## 8. 污染与发布门禁：verifier、数据和独立 evaluator 都要版本化

训练题与评测题按模板族/来源切分，不能只按题面字符串；verifier 隐藏测试不可进入 prompt。发布需比较冻结基线上的任务成功、hack ASR、格式率、长度、KL、成本，并抽样人工复核高影响 slice。


In [ ]:
@dataclass(frozen=True)
class RLVRArtifact:
    dataset_hash: str
    verifier_hash: str
    policy_base: str
    reward_recipe: str

def family_split(family_ids, holdout_families):
    return np.array([family not in holdout_families for family in family_ids])

# 同一模板族必须整体进入 train 或 holdout，制品摘要变化应可检测。
families = ["fraction", "fraction", "sorting", "geometry", "sorting"]
train_mask = family_split(families, {"sorting"})
assert train_mask.tolist() == [True, True, False, True, False]
artifact = RLVRArtifact("data-v3", "verify-v7", "base-v2", "correct+0.05format-hard-safety")
digest = hashlib.sha256(repr(artifact).encode()).hexdigest()
assert len(digest) == 64
assert digest != hashlib.sha256(repr(RLVRArtifact("data-v3", "verify-v8", "base-v2", artifact.reward_recipe)).encode()).hexdigest()


## 面试收束：怎样把实现讲成工程答案

建议按“目标与约束 → 数据/张量合同 → 核心公式 → 正确性 oracle → 性能与安全边界 → 发布门禁”作答。Notebook 里的小张量和受控状态机只证明机制成立，不等于真实集群吞吐、真实模型质量或生产安全性。上线前还要补齐目标硬件 profiling、故障注入、分布式一致性、真实数据切片、权限审计、版本化制品和回滚演练。

可继续追问：规模扩大后哪个状态最贵？哪条等价性可作为回归测试？输入或版本变化时怎样拒绝静默错误？指标改善是否只是成本、数据污染或评测器偏差造成的？
